In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

In [2]:
# carga de los datos para entrenamiento y test.
datos_entrenamiento = pd.read_csv('Sign Language MNIST/sign_mnist_train/sign_mnist_train.csv')
datos_prueba = pd.read_csv('Sign Language MNIST/sign_mnist_test/sign_mnist_test.csv')

In [3]:
print(f'''
datos entrenamiento:{datos_entrenamiento.info()}
''')


<class 'pandas.DataFrame'>
RangeIndex: 27455 entries, 0 to 27454
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 164.4 MB

datos entrenamiento:None



In [59]:
print(f'''
Datos de prueba: {datos_prueba.info()}''')

<class 'pandas.DataFrame'>
RangeIndex: 7172 entries, 0 to 7171
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 43.0 MB

Datos de prueba: None


In [4]:
# cantidad de pixeles 784 se obtiene el tamano de la imagen
print(f'Cantidad de pixeles {datos_entrenamiento.shape[1]-1}')
np.sqrt(784)

Cantidad de pixeles 784


np.float64(28.0)

In [5]:
# label, variable objetivo , verificamos los valores unicos para encontrar la cantidad de clases
clases = datos_entrenamiento['label'].unique()

In [56]:
# Separacion de datos para entramiento.
features_train = datos_entrenamiento.drop(columns='label')
label_train = datos_entrenamiento['label']

print(f'''Chequeo de dimensiones features and label
features : {features_train.shape}
label : {label_train.shape}''')

Chequeo de dimensiones features and label
features : (27455, 784)
label : (27455,)


In [55]:
# Separacion de datos para test
features_test = datos_entrenamiento.drop(columns='label')
label_test= datos_entrenamiento['label']
print(f'''
Chequeo de dimension para entrenamiento de test.
features_test : {features_test.shape}
label_test: {label_test}'''
)


Chequeo de dimension para entrenamiento de test.
features_test : (27455, 784)
label_test: 0         3
1         6
2         2
3         2
4        13
         ..
27450    13
27451    23
27452    18
27453    17
27454    23
Name: label, Length: 27455, dtype: int64


In [57]:
# Se procede a corroborar las caracteristicas y el valor de cada pixel con una intensidad de 0 a 255.
features_train.describe()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
count,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,...,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000,27455.000000
mean,145.419377,148.500273,151.247714,153.546531,156.210891,158.411255,160.472154,162.339683,163.954799,165.533673,...,141.104863,147.495611,153.325806,159.125332,161.969259,162.736696,162.906137,161.966454,161.137898,159.824731
std,41.358555,39.942152,39.056286,38.595247,37.111165,36.125579,35.016392,33.661998,32.651607,31.279244,...,63.751194,65.512894,64.427412,63.708507,63.738316,63.444008,63.509210,63.298721,63.610415,64.396846
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,121.000000,126.000000,130.000000,133.000000,137.000000,140.000000,142.000000,144.000000,146.000000,148.000000,...,92.000000,96.000000,103.000000,112.000000,120.000000,125.000000,128.000000,128.000000,128.000000,125.500000
50%,150.000000,153.000000,156.000000,158.000000,160.000000,162.000000,164.000000,165.000000,166.000000,167.000000,...,144.000000,162.000000,172.000000,180.000000,183.000000,184.000000,184.000000,182.000000,182.000000,182.000000
75%,174.000000,176.000000,178.000000,179.000000,181.000000,182.000000,183.000000,184.000000,185.000000,186.000000,...,196.000000,202.000000,205.000000,207.000000,208.000000,207.000000,207.000000,206.000000,204.000000,204.000000
max,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,...,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000


- En el siguiente analisis de EDA, se procedera a realiza una normalizacion de los valores de los pixeles de las imagenes.
Donde cada imagen esta compuesta por una dimension 784 pixeles, correspondiente a una dimension de 28x28, los valores de intensidad de los pixeles van desde los 0-255, por lo que seran normalizados en un rango de 0 a 1.

In [65]:
# funcion normalizar
def normalizar(imagenes, etiquetas):
    """toma las imagenes como caracteristicas y 
    sus etiquetas correspondientes a la clase de cada imagen.
    las imagenes se convierten a un tipo de datos flotante y sus valores de intensidad
    originalmente va desde 0 a 255 valores de intensidad y se normalizan a un rango de 0 a 1.
    Args:
        imagenes (float64): caracteristica de cada imagen representadas
        mediante sus valores de pixeles.
        etiquetas (float64): clase correspondiente a cada imagen.

    Returns:
        float64: retorna un tupla que contiene los valores de intensidad normalizados
        en un rango de 0 a 1 y las etiquetas quedan sin modificar.
    """    
    imagenes = tf.cast(imagenes,tf.float32)
    imagenes/=255
    return imagenes,etiquetas

In [66]:
# Normalizacion de los datos para entrenamiento y test
#train
normalized_train, label_train = normalizar(features_train,label_train)
#test
normalized_test , label_test = normalizar(features_test,label_test)

In [69]:
# al normalizar los datos originales, las caracteristicas se convierten en tensores,por lo tanto se procede a construir un dataset.
# para organizar las caracteristicas y etiquetas.
# dataset train
dataset_train = tf.data.Dataset.from_tensor_slices((normalized_train,label_train))
# dataset test
dataset_test = tf.data.Dataset.from_tensor_slices((normalized_test,label_test))